# 00. Databricks Setup and Reset (Bidirectional Demo)

Run this before every run of the bidirectional demo, and run it **after**
`00_snowflake_setup.sql`. Snowflake writes the Iceberg metadata; this notebook points
Unity Catalog at it.

What it does:

1. Registers `customers` and `orders` from the Snowflake-managed Iceberg files on S3.
2. Verifies the numbers match Snowflake.
3. Resets the demo: drops the Metric View and clears the Databricks sync state.

It leaves Databricks with **the same data as Snowflake and no semantic model**, which is
the starting position the demo depends on.

## Configuration

Catalog and schema mirror the Snowflake database and schema, so the table references
inside the Ossie file resolve on both platforms without rewriting.

In [ ]:
CATALOG   = "demos"
SCHEMA    = "ext_semantic_interop"
S3_BUCKET = "s3://snowflake-ossie-interop"     # <-- your bucket

METRIC_VIEW = f"{CATALOG}.{SCHEMA}.sales_metric_view"
STATE_DIR   = f"{S3_BUCKET}/ossie/_state/"

print(f"Catalog/schema : {CATALOG}.{SCHEMA}")
print(f"Metric View    : {METRIC_VIEW}")

## 1. Register the Iceberg tables

Snowflake-managed Iceberg tables are Parquet plus standard Iceberg metadata in your own
S3 bucket. Snowflake appends a random suffix to the base location, for example
`customers.6EWgT0se/`, so the paths are discovered rather than hardcoded.

Both platforms end up reading the same physical files. Nothing is copied.

In [ ]:
import re

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

table_paths = {}
for entry in dbutils.fs.ls(f"{S3_BUCKET}/iceberg/"):
    match = re.match(r"(customers|orders)\.\w+/$", entry.name)
    if match:
        table_paths[match.group(1)] = entry.path.rstrip("/")

if len(table_paths) < 2:
    raise Exception(
        f"Found only {sorted(table_paths)} under {S3_BUCKET}/iceberg/. "
        "Run 00_snowflake_setup.sql first."
    )

for table, path in table_paths.items():
    spark.sql(f"""
        CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.{table}
        AS SELECT * FROM read_files('{path}/data', format => 'parquet')
    """)
    print(f"Registered {CATALOG}.{SCHEMA}.{table}")
    print(f"           <- {path}")

## 2. Verify the numbers match Snowflake

In [ ]:
display(spark.sql(f"""
  SELECT c.region,
         SUM(o.order_amount) AS total_order_amount,
         COUNT(o.order_id)   AS order_count,
         SUM(o.order_qty)    AS total_quantity
    FROM {CATALOG}.{SCHEMA}.orders o
    JOIN {CATALOG}.{SCHEMA}.customers c USING (customer_id)
   GROUP BY c.region ORDER BY c.region
"""))
# expect EAST 750/5/12, WEST 700/5/11 -- identical to Snowflake

## 3. Reset the demo

Drops the Metric View so the demo can show it being created from nothing, and clears the
Databricks sync state so the first sync starts clean rather than comparing against a
fingerprint from a previous run.

Also pause any sync job left running in Workflows.

In [ ]:
spark.sql(f"DROP VIEW IF EXISTS {METRIC_VIEW}")
print(f"Dropped {METRIC_VIEW}")

try:
    dbutils.fs.rm(STATE_DIR, recurse=True)
    print(f"Cleared {STATE_DIR}")
except Exception as exc:
    print(f"No state to clear ({exc.__class__.__name__})")

display(spark.sql(f"SHOW VIEWS IN {CATALOG}.{SCHEMA}"))
# sales_metric_view should NOT be listed

## Ready

Databricks now has the data and no Metric View.

Open **`01_snowflake_semantic_view.ipynb`** in Snowflake to start the demo.